# Train YOLO26n locally on rover Unity data (aruco / hammer / mallet)

**Setup:** follow the repo `README.md` (build the dataset, then create the `.venv` environment).
Then pick the `.venv` interpreter as the notebook kernel and run all cells.

Runs go to `YOLO_train+val/runs/`, final weights to `YOLO_train+val/weights/`.
If `best.pt` already exists, training is skipped. To continue an interrupted run set `RESUME = True`.

In [ ]:
# ---------------- config ----------------
from pathlib import Path
import torch

# Works whether the kernel starts in the repo root or in the dataset folder.
DATASET = next((p.resolve() for p in (Path("."), Path("YOLO_train+val")) if (p / "train.yaml").exists()), None)
assert DATASET, "train.yaml not found: start Jupyter from the repo root or from YOLO_train+val/"

DEVICE        = 0 if torch.cuda.is_available() else "cpu"
IMGSZ         = 1280      # native width of the renders; 640 is much faster but loses small objects
EPOCHS        = 100
BATCH         = 2         # keep small to limit RAM; raise if you have headroom
WORKERS       = 2         # each data-loader worker costs extra RAM
FRACTION      = 1.0       # share of the training images to use; e.g. 0.02 for a quick trial run
RUN_NAME      = f"yolo26n_{IMGSZ}_local"
RUNS_DIR      = DATASET / "runs"
WEIGHTS_DIR   = DATASET / "weights"
MODEL         = str(WEIGHTS_DIR / "yolo26n.pt")   # downloaded here on first use
FORCE_RETRAIN = False     # True = train again even if best.pt already exists
RESUME        = False     # True = continue an interrupted run from last.pt

RUNS_DIR.mkdir(exist_ok=True); WEIGHTS_DIR.mkdir(exist_ok=True)
print("dataset:", DATASET, "| device:", DEVICE)

Check the file counts against `split_manifest.csv`, and write a data YAML with this machine's absolute path.

In [ ]:
import csv, yaml
from collections import Counter

expected = Counter(r["split"] for r in csv.DictReader(open(DATASET / "split_manifest.csv")))
for split in ("train", "val", "test"):
    n_img = len(list((DATASET / "images" / split).glob("*.jpg")))
    n_lbl = len(list((DATASET / "labels" / split).glob("*.txt")))
    print(f"{split:5s}: {n_img} images, {n_lbl} labels (expected {expected[split]})")
    assert n_img == n_lbl == expected[split], "dataset incomplete: re-run scripts/build_yolo_dataset.py"

cfg = yaml.safe_load(open(DATASET / "train.yaml"))
cfg["path"] = str(DATASET)
names = cfg["names"]
DATA_YAML = str(RUNS_DIR / "data_local.yaml")
yaml.safe_dump(cfg, open(DATA_YAML, "w"), sort_keys=False)

Sanity check: draw ground-truth boxes on a few random training images.

In [ ]:
import random, cv2
import matplotlib.pyplot as plt

colors = {0: (255, 0, 0), 1: (0, 255, 0), 2: (0, 128, 255)}
imgs = random.sample(sorted(Path(DATASET, "images/train").glob("*.jpg")), 4)
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, p in zip(axes.flat, imgs):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    for line in Path(DATASET, "labels/train", p.stem + ".txt").read_text().splitlines():
        c, cx, cy, bw, bh = line.split(); c = int(c)
        cx, cy, bw, bh = float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
        cv2.rectangle(im, (int(cx - bw / 2), int(cy - bh / 2)), (int(cx + bw / 2), int(cy + bh / 2)), colors[c], 2)
        cv2.putText(im, names[c], (int(cx - bw / 2), int(cy - bh / 2) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, colors[c], 2)
    ax.imshow(im); ax.set_title(p.name); ax.axis("off")
plt.tight_layout(); plt.show()

## Train
Skipped when `best.pt` already exists (unless `FORCE_RETRAIN = True`).

On CPU this is very slow. Try `FRACTION = 0.02`, `EPOCHS = 1` and a different `RUN_NAME` first to see how long an epoch takes.

In [ ]:
from ultralytics import YOLO

run_dir = RUNS_DIR / RUN_NAME
best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

if RESUME:
    assert last_pt.exists(), f"nothing to resume: {last_pt} missing"
    YOLO(str(last_pt)).train(resume=True)
elif best_pt.exists() and not FORCE_RETRAIN:
    print(f"Found existing weights, skipping training: {best_pt}")
else:
    YOLO(MODEL).train(
        data=DATA_YAML,
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=BATCH,
        workers=WORKERS,
        device=DEVICE,
        fraction=FRACTION,
        cache=False,     # do not hold images in RAM
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
    )

Copy the final model to a fixed location.

In [ ]:
import shutil

final_pt = WEIGHTS_DIR / f"{RUN_NAME}_best.pt"
shutil.copy2(best_pt, final_pt)
print("Final model saved to:", final_pt)
model = YOLO(str(final_pt))

## Evaluate on the held-out test split

In [ ]:
metrics = model.val(data=DATA_YAML, split="test", imgsz=IMGSZ,
                    batch=BATCH, workers=WORKERS, device=DEVICE,
                    project=str(RUNS_DIR), name=f"{RUN_NAME}_test", exist_ok=True)
print(f"test mAP50    : {metrics.box.map50:.4f}")
print(f"test mAP50-95 : {metrics.box.map:.4f}")
for i, c in enumerate(metrics.box.ap_class_index):
    print(f"  {names[int(c)]:8s} mAP50={metrics.box.ap50[i]:.4f}  mAP50-95={metrics.box.ap[i]:.4f}")

## Predict on a few test images

In [ ]:
test_imgs = random.sample(sorted(Path(DATASET, "images/test").glob("*.jpg")), 4)
results = model.predict([str(p) for p in test_imgs], imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, r in zip(axes.flat, results):
    ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(Path(r.path).name); ax.axis("off")
plt.tight_layout(); plt.show()

Optional: export to ONNX (written next to the `.pt`).

In [ ]:
# onnx_path = model.export(format="onnx", imgsz=IMGSZ)
# print(onnx_path)